# Phase II_01: Spectral Relabeling with USGS MTBS Standards

**Objective**: Create multi-class burn labels using official USGS burn severity classifications

**Input**: CaBuAr dataset with original binary labels

**Output**: Multi-class labels (7 classes) based on Normalized Burn Ratio (NBR)

**Reference**: `docs/phase3_relabeling/SPECTRAL_RELABELING_STRATEGY.md`

## Setup: Google Drive and Dependencies

In [ ]:
import sys
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
import json

sys.path.insert(0, '/content/RETINNA/src')

# Mount Google Drive
print("📁 Initializing Google Drive...")
from drive_utils import setup_drive_for_colab

drive_manager = setup_drive_for_colab(verbose=True)

if drive_manager is None:
    raise RuntimeError("Failed to connect to Google Drive - aborting")

print("\n" + "="*70)
print("PHASE II_01: SPECTRAL RELABELING WITH USGS MTBS STANDARDS")
print("="*70)
print(f"Session start: {datetime.now().isoformat()}")
print(f"Drive cache: {drive_manager.root}")

## Load Dataset

In [ ]:
from dataset import get_dataloaders

print("\n📊 Loading CaBuAr dataset...")
dataloaders = get_dataloaders(batch_size=1, num_workers=0, normalize=True, root=None)

train_dataset = dataloaders['datasets']['train']
val_dataset = dataloaders['datasets']['val']
test_dataset = dataloaders['datasets']['test']

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples: {len(val_dataset)}")
print(f"✓ Test samples: {len(test_dataset)}")
print(f"✓ Total: {len(train_dataset) + len(val_dataset) + len(test_dataset)} samples")

## Spectral Analysis: Compute USGS NBR and Classification

In [ ]:
def compute_spectral_indices(pre_fire, post_fire):
    """
    Compute spectral indices for burn classification.
    
    Args:
        pre_fire: [12, H, W] pre-fire Sentinel-2 bands
        post_fire: [12, H, W] post-fire Sentinel-2 bands
    
    Returns:
        dict with NBR, dNBR, MNDWI, etc.
    """
    # Band indices: B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12, CLP, SCL
    red_idx, nir_idx, swir1_idx, green_idx = 2, 6, 10, 1
    
    # Extract bands
    red_pre = pre_fire[red_idx]
    red_post = post_fire[red_idx]
    nir_pre = pre_fire[nir_idx]
    nir_post = post_fire[nir_idx]
    swir_pre = pre_fire[swir1_idx]
    swir_post = post_fire[swir1_idx]
    green_pre = pre_fire[green_idx]
    green_post = post_fire[green_idx]
    
    eps = 1e-8
    
    # NBR = (NIR - SWIR) / (NIR + SWIR)
    nbr_pre = (nir_pre - swir_pre) / (nir_pre + swir_pre + eps)
    nbr_post = (nir_post - swir_post) / (nir_post + swir_post + eps)
    
    # dNBR = NBRpre - NBRpost (positive = burned)
    dnbr = nbr_pre - nbr_post
    
    # MNDWI = (Green - SWIR) / (Green + SWIR) for water detection
    mndwi = (green_post - swir_post) / (green_post + swir_post + eps)
    
    # NDVI for context
    ndvi_pre = (nir_pre - red_pre) / (nir_pre + red_pre + eps)
    ndvi_post = (nir_post - red_post) / (nir_post + red_post + eps)
    
    return {
        'nbr_pre': nbr_pre,
        'nbr_post': nbr_post,
        'dnbr': dnbr,
        'mndwi': mndwi,
        'ndvi_pre': ndvi_pre,
        'ndvi_post': ndvi_post
    }


def classify_pixel(dnbr, mndwi, scl=None):
    """
    Classify pixel using USGS MTBS thresholds.
    
    Classes:
        0: Unburned (dNBR >= 0.27)
        1: Low Severity (0.05 < dNBR <= 0.27)
        2: Moderate Severity (-0.1 < dNBR <= 0.05)
        3: High Severity (-0.27 < dNBR <= -0.1)
        4: Extreme Severity (dNBR < -0.27)
        5: Water (MNDWI > 0.3)
        6: Cloud/Shadow/Invalid
    """
    # Priority: Cloud > Water > Burn classes
    
    if scl is not None:
        # Sentinel-2 SCL cloud/shadow classes: 0,1,2,3,8,9,10,11
        if scl in [0, 1, 2, 3, 8, 9, 10, 11]:
            return 6
    
    if mndwi > 0.3:
        return 5
    
    if dnbr < -0.27:
        return 4  # Extreme
    elif -0.27 <= dnbr <= -0.1:
        return 3  # High
    elif -0.1 < dnbr <= 0.05:
        return 2  # Moderate
    elif 0.05 < dnbr <= 0.27:
        return 1  # Low
    else:
        return 0  # Unburned


print("✓ Spectral analysis functions defined")

## Process All Samples: Train + Val + Test

In [ ]:
def process_dataset(dataset, dataset_name, start_idx=0):
    """
    Process entire dataset and generate multi-class labels.
    
    Returns:
        labels: [N_samples, 512, 512] multi-class labels (0-6)
        metrics: dict with statistics
    """
    print(f"\n📊 Processing {dataset_name} set ({len(dataset)} samples)...")
    
    all_labels = []
    all_dnbr = []
    all_mndwi = []
    
    class_counts = {i: 0 for i in range(7)}
    
    for sample_idx in range(len(dataset)):
        if (sample_idx + 1) % max(1, len(dataset) // 10) == 0:
            print(f"  {sample_idx + 1}/{len(dataset)} samples processed...")
        
        sample = dataset[sample_idx]
        image = sample['image'].numpy()  # [2, 12, 512, 512]
        mask_original = sample['mask'].numpy()[0]  # [512, 512] binary
        
        pre_fire = image[0]
        post_fire = image[1]
        
        # Compute indices
        indices = compute_spectral_indices(pre_fire, post_fire)
        dnbr = indices['dnbr']
        mndwi = indices['mndwi']
        
        # Classify each pixel
        labels = np.zeros((512, 512), dtype=np.uint8)
        for y in range(512):
            for x in range(512):
                labels[y, x] = classify_pixel(dnbr[y, x], mndwi[y, x])
        
        # Track statistics
        all_labels.append(labels)
        all_dnbr.append(dnbr)
        all_mndwi.append(mndwi)
        
        for class_id in range(7):
            class_counts[class_id] += (labels == class_id).sum()
    
    # Stack all labels
    labels_array = np.stack(all_labels, axis=0)  # [N, 512, 512]
    
    # Compute metrics
    total_pixels = labels_array.size
    metrics = {
        'dataset': dataset_name,
        'n_samples': len(dataset),
        'total_pixels': int(total_pixels),
        'class_distribution': {}
    }
    
    for class_id in range(7):
        class_name = [
            'Unburned',
            'Low Severity',
            'Moderate Severity',
            'High Severity',
            'Extreme Severity',
            'Water',
            'Cloud/Shadow'
        ][class_id]
        
        count = int(class_counts[class_id])
        pct = 100.0 * count / total_pixels
        
        metrics['class_distribution'][class_name] = {
            'pixels': count,
            'percentage': float(f"{pct:.2f}")
        }
        
        print(f"    Class {class_id} ({class_name:20}): {count:12,} px ({pct:5.2f}%)")
    
    return labels_array, metrics


# Process all splits
train_labels, train_metrics = process_dataset(train_dataset, "Train")
val_labels, val_metrics = process_dataset(val_dataset, "Val", start_idx=len(train_dataset))
test_labels, test_metrics = process_dataset(test_dataset, "Test", start_idx=len(train_dataset)+len(val_dataset))

print("\n✓ All datasets processed")

## Combine and Save Results

In [ ]:
# Combine splits
all_labels = np.concatenate([train_labels, val_labels, test_labels], axis=0)
all_metrics = {
    'timestamp': datetime.now().isoformat(),
    'total_samples': len(train_dataset) + len(val_dataset) + len(test_dataset),
    'splits': {
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics
    }
}

print(f"\n💾 Saving results...")
print(f"  Labels shape: {all_labels.shape}")
print(f"  Total samples: {all_metrics['total_samples']}")

# Save multi-class labels to Drive
labels_path = drive_manager.save_with_timestamp(
    torch.from_numpy(all_labels),
    rel_path="phase2/II_01_relabeling",
    filename_base="multi_class_labels",
    file_format=".pt",
    verbose=True
)

# Save metrics to Drive
metrics_path = drive_manager.save_with_timestamp(
    all_metrics,
    rel_path="phase2/II_01_relabeling",
    filename_base="metrics",
    file_format=".json",
    verbose=True
)

if labels_path and metrics_path:
    print(f"\n✓ Results saved to Drive")
    print(f"  Labels: {labels_path.name}")
    print(f"  Metrics: {metrics_path.name}")
else:
    print(f"\n✗ Failed to save to Drive")

## Quality Control: Verify Results

In [ ]:
# Reload from Drive to verify
print("\n🔍 Quality Control: Verifying saved files...")

latest_labels = drive_manager.get_latest_file(
    rel_path="phase2/II_01_relabeling",
    filename_pattern="multi_class_labels"
)

if latest_labels:
    loaded_labels = torch.load(latest_labels)
    print(f"✓ Labels verified: {loaded_labels.shape}")
    
    # Check class distribution
    print(f"\nClass distribution in loaded labels:")
    class_names = [
        'Unburned',
        'Low Severity',
        'Moderate Severity',
        'High Severity',
        'Extreme Severity',
        'Water',
        'Cloud/Shadow'
    ]
    
    for class_id in range(7):
        count = (loaded_labels == class_id).sum().item()
        total = loaded_labels.numel()
        pct = 100.0 * count / total
        print(f"  {class_id}: {class_names[class_id]:20} {count:12,} px ({pct:5.2f}%)")
else:
    print(f"✗ Could not verify saved labels")

print(f"\n✓ Phase II_01 complete at {datetime.now().isoformat()}")
print(f"\nNext: II_02_rgb_ir_training.ipynb")